# Redimensionamento Seletivo de Fotos da Obra

**Objetivo:** redimensionar apenas as imagens que foram identificadas como necessitando de redimensionamento no relatório CSV anterior, com base na análise de pré-processamento.

Este notebook:

1. Lê o relatório `relatorio_preprocessamento.csv`;
2. Identifica as imagens marcadas como `TRUE` na coluna `Redimensionamento`;
3. Redimensiona apenas essas imagens para o tamanho alvo;
4. Salva os arquivos na pasta `output`;
5. Gera um relatório simples do processamento.

**Autora:** Adriana Rolim  
**Versão:** 2.0 — Redimensionamento Seletivo  
**Data:** Agosto de 2025

## 1. Importação das bibliotecas e montagem do Google Drive

Execute esta célula primeiro para permitir que o Colab acesse os arquivos salvos no Google Drive.

In [ ]:
import os
import cv2
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

## 2. Configurações principais

Ajuste os caminhos abaixo caso sua estrutura de pastas esteja diferente.

In [ ]:
# ============================
# CONFIGURAÇÕES DO NOTEBOOK
# ============================

PASTA_BASE = '/content/drive/MyDrive/Python_VC'

CSV_RELATORIO = os.path.join(PASTA_BASE, "relatorio_preprocessamento.csv")
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output")

# Tamanho final desejado: largura x altura
TAMANHO_ALVO = (1920, 1080)

print("Configurações carregadas:")
print(f"📁 Pasta base: {PASTA_BASE}")
print(f"📊 CSV relatório: {CSV_RELATORIO}")
print(f"📁 Pasta de entrada: {PASTA_ENTRADA}")
print(f"📁 Pasta de saída: {PASTA_SAIDA}")
print(f"📐 Tamanho alvo: {TAMANHO_ALVO[0]}x{TAMANHO_ALVO[1]}")

## 3. Função para verificar a configuração

Esta etapa verifica se a pasta base, o CSV e a pasta de entrada existem.

In [ ]:
def verificar_configuracao():
    """Verifica se todos os arquivos e pastas necessários existem."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO:")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📊 Arquivo CSV: {CSV_RELATORIO}")
    print(f"📁 Pasta entrada: {PASTA_ENTRADA}")
    print(f"📁 Pasta saída: {PASTA_SAIDA}")
    print(f"📐 Tamanho alvo: {TAMANHO_ALVO[0]}x{TAMANHO_ALVO[1]}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada!")

    if not os.path.exists(CSV_RELATORIO):
        problemas.append("❌ Arquivo CSV de relatório não encontrado!")
        problemas.append("💡 Execute primeiro o script de análise exploratória.")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada!")

    if problemas:
        print("\n".join(problemas))
        return False

    print("✅ Configuração verificada com sucesso.")
    return True


config_ok = verificar_configuracao()

## 4. Leitura do CSV e identificação das imagens a redimensionar

O notebook espera que o CSV contenha pelo menos as colunas:

- `Imagem`
- `Redimensionamento`

A coluna `Redimensionamento` deve conter valores equivalentes a verdadeiro, como `True`, `TRUE`, `1`, `Sim` ou `SIM`.

In [ ]:
def normalizar_booleano(valor):
    """Converte valores variados para booleano."""
    if pd.isna(valor):
        return False

    if isinstance(valor, bool):
        return valor

    texto = str(valor).strip().lower()

    valores_verdadeiros = {"true", "1", "sim", "s", "yes", "y", "verdadeiro"}
    return texto in valores_verdadeiros


def ler_imagens_para_redimensionar(csv_path):
    """
    Lê o CSV e identifica quais imagens precisam de redimensionamento.

    Parâmetros:
        csv_path (str): caminho para o arquivo CSV

    Retorno:
        list: lista de nomes de arquivos que precisam de redimensionamento
    """
    print("\n📊 LENDO RELATÓRIO CSV...")

    try:
        df = pd.read_csv(csv_path)
        print(f"✅ CSV carregado com {len(df)} imagens analisadas.")

        colunas_obrigatorias = ["Imagem", "Redimensionamento"]
        colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]

        if colunas_ausentes:
            print(f"❌ Colunas obrigatórias ausentes no CSV: {colunas_ausentes}")
            print("📋 Colunas disponíveis:", list(df.columns))
            return [], df

        df["Redimensionamento_bool"] = df["Redimensionamento"].apply(normalizar_booleano)

        imagens_para_redimensionar = (
            df.loc[df["Redimensionamento_bool"], "Imagem"]
            .dropna()
            .astype(str)
            .tolist()
        )

        print("\n📈 Estatísticas do CSV:")
        print(f"   - Total de imagens analisadas: {len(df)}")
        print(f"   - Precisam de redimensionamento: {len(imagens_para_redimensionar)}")
        print(f"   - Não precisam de redimensionamento: {len(df) - len(imagens_para_redimensionar)}")

        if imagens_para_redimensionar:
            print("\n📝 Primeiras imagens que serão redimensionadas:")
            for img in imagens_para_redimensionar[:10]:
                print(f"   ✅ {img}")

            if len(imagens_para_redimensionar) > 10:
                print(f"   ... e mais {len(imagens_para_redimensionar) - 10} imagens.")
        else:
            print("ℹ️ Nenhuma imagem precisa de redimensionamento segundo o CSV.")

        return imagens_para_redimensionar, df

    except Exception as e:
        print(f"❌ Erro ao ler o CSV: {str(e)}")
        return [], None


if config_ok:
    imagens_para_processar, df_relatorio = ler_imagens_para_redimensionar(CSV_RELATORIO)
else:
    imagens_para_processar, df_relatorio = [], None

## 5. Visualização rápida do relatório

Esta célula mostra as primeiras linhas do CSV carregado, caso ele tenha sido lido corretamente.

In [ ]:
if df_relatorio is not None:
    display(df_relatorio.head())
else:
    print("⚠️ Nenhum relatório carregado para visualização.")

## 6. Funções de processamento

Estas funções criam a pasta de saída, redimensionam mantendo proporção e aplicam `letterbox` com fundo preto quando necessário.

In [ ]:
def garantir_pasta(pasta):
    """Cria a pasta se ela não existir."""
    os.makedirs(pasta, exist_ok=True)
    print(f"📁 Pasta garantida: {pasta}")


def redimensionar_imagem(img, tamanho_alvo, nome_arquivo):
    """Redimensiona mantendo a proporção e aplica letterbox se necessário."""
    largura_alvo, altura_alvo = tamanho_alvo
    h, w = img.shape[:2]

    print(f"   📐 {nome_arquivo}: {w}x{h} → {largura_alvo}x{altura_alvo}")

    proporcao = min(largura_alvo / w, altura_alvo / h)
    nova_largura = int(w * proporcao)
    nova_altura = int(h * proporcao)

    img_redimensionada = cv2.resize(
        img,
        (nova_largura, nova_altura),
        interpolation=cv2.INTER_AREA
    )

    imagem_final = cv2.copyMakeBorder(
        img_redimensionada,
        top=(altura_alvo - nova_altura) // 2,
        bottom=(altura_alvo - nova_altura + 1) // 2,
        left=(largura_alvo - nova_largura) // 2,
        right=(largura_alvo - nova_largura + 1) // 2,
        borderType=cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )

    return imagem_final

## 7. Processamento seletivo das imagens

Esta célula percorre apenas as imagens marcadas para redimensionamento no CSV.

In [ ]:
def processar_imagens_seletivamente(pasta_entrada, pasta_saida, lista_imagens, tamanho_alvo):
    """
    Processa apenas as imagens da lista, redimensionando-as.

    Parâmetros:
        pasta_entrada (str): pasta com imagens originais
        pasta_saida (str): pasta para salvar imagens processadas
        lista_imagens (list): lista de nomes de arquivos para processar
        tamanho_alvo (tuple): tamanho alvo no formato largura, altura

    Retorno:
        tuple: quantidade processada, lista de processadas, lista de não encontradas, lista de erros
    """
    garantir_pasta(pasta_saida)

    if not lista_imagens:
        print("ℹ️ Nenhuma imagem para processar.")
        return 0, [], [], []

    print(f"\n🔄 PROCESSANDO {len(lista_imagens)} IMAGENS...")
    print("=" * 60)

    processadas = []
    nao_encontradas = []
    erros = []

    for arquivo in lista_imagens:
        caminho_entrada = os.path.join(pasta_entrada, arquivo)

        if not os.path.exists(caminho_entrada):
            print(f"❌ Imagem não encontrada: {arquivo}")
            nao_encontradas.append(arquivo)
            continue

        try:
            img = cv2.imread(caminho_entrada)

            if img is None:
                print(f"❌ Não foi possível abrir: {arquivo}")
                erros.append((arquivo, "Não foi possível abrir a imagem."))
                continue

            img_redimensionada = redimensionar_imagem(img, tamanho_alvo, arquivo)

            caminho_saida = os.path.join(pasta_saida, arquivo)

            if arquivo.lower().endswith((".jpg", ".jpeg")):
                sucesso = cv2.imwrite(
                    caminho_saida,
                    img_redimensionada,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
            else:
                sucesso = cv2.imwrite(caminho_saida, img_redimensionada)

            if sucesso:
                print(f"   ✅ Salvo: {arquivo}")
                processadas.append(arquivo)
            else:
                print(f"❌ Falha ao salvar: {arquivo}")
                erros.append((arquivo, "Falha ao salvar o arquivo."))

        except Exception as e:
            print(f"❌ Erro ao processar {arquivo}: {str(e)}")
            erros.append((arquivo, str(e)))

    print("\n" + "=" * 60)
    print("📊 RESUMO DO PROCESSAMENTO SELETIVO:")
    print(f"   ✅ Imagens processadas com sucesso: {len(processadas)}")
    print(f"   ❌ Imagens não encontradas: {len(nao_encontradas)}")
    print(f"   ❌ Erros de processamento: {len(erros)}")
    print(f"   📁 Pasta de saída: {pasta_saida}")

    return len(processadas), processadas, nao_encontradas, erros

## 8. Geração do relatório do processamento

O relatório será salvo como `relatorio_redimensionamento.txt` dentro da pasta `output`.

In [ ]:
def gerar_relatorio_processamento(
    pasta_saida,
    lista_processadas,
    total_necessitavam,
    lista_nao_encontradas=None,
    lista_erros=None
):
    """Gera um relatório simples do processamento."""
    lista_nao_encontradas = lista_nao_encontradas or []
    lista_erros = lista_erros or []

    garantir_pasta(pasta_saida)

    relatorio_path = os.path.join(pasta_saida, "relatorio_redimensionamento.txt")

    with open(relatorio_path, "w", encoding="utf-8") as f:
        f.write("RELATÓRIO DE REDIMENSIONAMENTO SELETIVO\n")
        f.write("=" * 50 + "\n")
        f.write(f"Data: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Tamanho alvo: {TAMANHO_ALVO[0]}x{TAMANHO_ALVO[1]}\n")
        f.write(f"Total que necessitavam redimensionamento: {total_necessitavam}\n")
        f.write(f"Total processadas com sucesso: {len(lista_processadas)}\n")
        f.write(f"Total não encontradas: {len(lista_nao_encontradas)}\n")
        f.write(f"Total com erro: {len(lista_erros)}\n")

        f.write("\nIMAGENS PROCESSADAS:\n")
        for img in lista_processadas:
            f.write(f"- {img}\n")

        if lista_nao_encontradas:
            f.write("\nIMAGENS NÃO ENCONTRADAS:\n")
            for img in lista_nao_encontradas:
                f.write(f"- {img}\n")

        if lista_erros:
            f.write("\nERROS DE PROCESSAMENTO:\n")
            for img, erro in lista_erros:
                f.write(f"- {img}: {erro}\n")

    print(f"📄 Relatório salvo: {relatorio_path}")

    return relatorio_path

## 9. Executar o redimensionamento seletivo

Execute esta célula para rodar o fluxo completo após confirmar que as configurações estão corretas.

In [ ]:
print("🚀 INICIANDO REDIMENSIONAMENTO SELETIVO")
print("=" * 50)

if config_ok and imagens_para_processar:
    total_processadas, imagens_processadas, imagens_nao_encontradas, erros_processamento = (
        processar_imagens_seletivamente(
            PASTA_ENTRADA,
            PASTA_SAIDA,
            imagens_para_processar,
            TAMANHO_ALVO
        )
    )

    if total_processadas > 0:
        caminho_relatorio = gerar_relatorio_processamento(
            PASTA_SAIDA,
            imagens_processadas,
            len(imagens_para_processar),
            imagens_nao_encontradas,
            erros_processamento
        )

        print("\n🎯 PROCESSAMENTO CONCLUÍDO!")
        print(f"   📁 Imagens redimensionadas salvas em: {PASTA_SAIDA}")
        print(f"   📄 Relatório salvo em: {caminho_relatorio}")
        print(f"   💡 {total_processadas} imagem(ns) foram processadas de {len(imagens_para_processar)} necessária(s).")
    else:
        print("\n⚠️ Nenhuma imagem foi processada com sucesso.")

elif config_ok and not imagens_para_processar:
    print("\nℹ️ Nenhuma imagem necessita de redimensionamento.")
    print("💡 Todas as imagens já estão no tamanho adequado, conforme o CSV.")

else:
    print("\n❌ Não foi possível iniciar o processamento seletivo.")
    print("\n💡 SOLUÇÃO:")
    print("1. Execute primeiro o script de análise exploratória.")
    print("2. Certifique-se de que o arquivo 'relatorio_preprocessamento.csv' foi gerado.")
    print("3. Verifique se a pasta 'fotos_obra' contém as imagens originais.")

## 10. Conferência dos arquivos gerados

Esta célula lista alguns arquivos gerados na pasta de saída.

In [ ]:
if os.path.exists(PASTA_SAIDA):
    arquivos_saida = os.listdir(PASTA_SAIDA)
    print(f"📁 Total de arquivos na pasta de saída: {len(arquivos_saida)}")
    print("\nPrimeiros arquivos encontrados:")
    for arquivo in arquivos_saida[:20]:
        print(f"- {arquivo}")
else:
    print("⚠️ A pasta de saída ainda não existe.")

## 11. Opcional: compactar a pasta de saída

Use esta célula se quiser gerar um arquivo `.zip` com as imagens processadas.

In [ ]:
import shutil

GERAR_ZIP = False  # Altere para True se quiser compactar a pasta output

if GERAR_ZIP:
    caminho_zip = shutil.make_archive(PASTA_SAIDA, 'zip', PASTA_SAIDA)
    print(f"✅ Arquivo ZIP criado: {caminho_zip}")
else:
    print("ℹ️ Compactação não executada. Para gerar o ZIP, altere GERAR_ZIP para True e execute novamente.")